In [1]:
from pathlib import Path
import os
import time
import random
import threading
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, roc_auc_score, f1_score
)

from models_transformers import TwoHeadTransformerNet, SingleOutTransformerNet
# from sklearn.decomposition import PCA

In [9]:
# %% [TRAIN ALL MULTI] Train M1, M2, M3 + save metrics + profile
EPOCHS = 20
LR = 1e-3
WD = 1e-6

# weights
LAMBDA_AGE = 1.0
LAMBDA_METS = 0.5
LAMBDA_SEX = 0.1

# M3 weights
LAMBDA_AGE_CONT = 1.0
LAMBDA_AGE_BIN = 0.8
LAMBDA_METS = 0.5
LAMBDA_SEX = 0.1

multi_test_rows_all = []

# --- Train M3 (kept as in original pipeline; NOT used for single-output or SHAP per your request)
def _train_m3_call():
    return train_3out(
        model_tr, train_loader, val_loader,
        tag="M3_multi_transformer",
        lr=LR, weight_decay=WD, epochs=EPOCHS,
        lambda_age=LAMBDA_AGE, lambda_mets=LAMBDA_METS, lambda_sex=LAMBDA_SEX,
        metrics_every=10, sex_threshold=0.5
    )


hist_m3, t_m3, pk_m3, st_m3 = profile_training(_train_m3_call, poll_every_sec=0.05)
profile_rows.append({
    "model_tag": "M3_multi_transformer",
    "train_seconds": t_m3,
    "start_rss_mb": st_m3,
    "peak_rss_mb": pk_m3,
    "rss_increase_mb": (pk_m3 - st_m3) if (not np.isnan(pk_m3) and not np.isnan(st_m3)) else np.nan
})
torch.save(model_tr.state_dict(), OUTDIR / "model_M3_transformer.pt")
# torch.save(model_proj3.state_dict(), OUTDIR / "model_M3_transformer.pt")
print(f"[PROFILE M3] time={t_m3:.2f}s | start_rss={st_m3:.1f}MB | peak_rss={pk_m3:.1f}MB")

m3_test = eval_split_metrics_3out_direct(model_tr, X_test_s, age_test, mets_test, 
                                         sex_test, sex_threshold=0.5)
multi_test_rows_all += [
    ["M1_multi", "test", "age", m3_test["age_R2"], m3_test["age_RMSE"], m3_test["age_MAE"], 
     np.nan, np.nan, np.nan],
    ["M1_multi", "test", "MetSCORE", m3_test["MetSCORE_R2"], m3_test["MetSCORE_RMSE"], 
     m3_test["MetSCORE_MAE"], np.nan, np.nan, np.nan],
    ["M1_multi", "test", "sex", np.nan, np.nan, np.nan, m3_test["sex_ACC"], 
     m3_test["sex_AUC"], m3_test["sex_F1"]],
]

metrics_multi_all_df = pd.DataFrame(
    multi_test_rows_all,
    columns=["model_type", "split", "output", "R2", "RMSE", "MAE", "ACC", "AUC", "F1"]
)
metrics_multi_all_path = OUTDIR / "metrics_test_multi_outputs_M1_M2_M3.csv"
metrics_multi_all_df.to_csv(metrics_multi_all_path, index=False, float_format="%.6f")
print("\nSaved:", metrics_multi_all_path)
print(metrics_multi_all_df)

[M3_multi_transformer] ep=001 best_val=inf lr=1.00e-03 | loss va=0.2954 (age=0.1925, mets=0.1320, sex=0.3687)
   TRAIN: age(R2=0.590, RMSE=8.643, MAE=6.790) | MetSCORE(R2=0.726, RMSE=0.133, MAE=0.095) | sex(ACC=0.856, AUC=0.921, F1=0.878)
   VAL  : age(R2=0.585, RMSE=8.669, MAE=6.695) | MetSCORE(R2=0.720, RMSE=0.136, MAE=0.096) | sex(ACC=0.868, AUC=0.931, F1=0.888)
[M3_multi_transformer] ep=010 best_val=0.207707 lr=1.00e-03 | loss va=0.2196 (age=0.1657, mets=0.0786, sex=0.1458)
   TRAIN: age(R2=0.687, RMSE=7.552, MAE=5.875) | MetSCORE(R2=0.850, RMSE=0.098, MAE=0.067) | sex(ACC=0.948, AUC=0.987, F1=0.955)
   VAL  : age(R2=0.644, RMSE=8.030, MAE=6.159) | MetSCORE(R2=0.842, RMSE=0.102, MAE=0.070) | sex(ACC=0.952, AUC=0.987, F1=0.958)
[M3_multi_transformer] ep=020 best_val=0.207707 lr=5.00e-04 | loss va=0.2120 (age=0.1588, mets=0.0763, sex=0.1501)
   TRAIN: age(R2=0.748, RMSE=6.776, MAE=5.275) | MetSCORE(R2=0.862, RMSE=0.094, MAE=0.067) | sex(ACC=0.955, AUC=0.989, F1=0.962)
   VAL  : age(R

In [10]:
# %% [SINGLE OUTPUT] Train single-output baselines using M2 architecture (MLP trunk) + profiling
# MOD: single-output models now share the same "M2 trunk style" (MLP with ReLU) rather than a separate MLP builder.

import copy


# class SingleOutNet_M2(nn.Module):
#     """
#     Single-output model that matches M2's trunk definition: build_mlp(in_dim, HIDDEN) + linear head -> 1 logit/value.
#     This is the requested "single-output using model M2 (not M3)".
#     """
#     def __init__(self, in_dim, hidden=HIDDEN):
#         super().__init__()
#         self.trunk, trunk_out = build_mlp(in_dim, hidden)
#         self.head = nn.Linear(trunk_out, 1)

#     def forward(self, x):
#         h = self.trunk(x)
#         return self.head(h)


@torch.no_grad()
def predict_np_single(model, Xn):
    model.eval()
    xb = torch.from_numpy(Xn).to(DEVICE)
    return model(xb).cpu().numpy()


def run_epoch_single(model, loader, kind, optimizer=None):
    reg_loss = nn.SmoothL1Loss()
    bce = nn.BCEWithLogitsLoss()

    train = optimizer is not None
    model.train(train)

    total, n = 0.0, 0
    for xb, agec, mets, sex in loader:
    # for xb, agec, agebin, mets, sex in loader:
        xb = xb.to(DEVICE)
        agec = agec.to(DEVICE)
        mets = mets.to(DEVICE)
        sex = sex.to(DEVICE)

        out = model(xb)  # (n,1)

        if kind == "age":
            loss = reg_loss(out, agec)
        elif kind == "mets":
            loss = reg_loss(out, mets)
        elif kind == "sex":
            loss = bce(out, sex)
        else:
            raise ValueError(kind)

        if train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        bs = xb.size(0)
        total += loss.item() * bs
        n += bs

    return total / max(n, 1)


def eval_single_metrics(model, kind, Xn, y_age_true, y_mets_true, y_sex_true, sex_threshold=0.5):
    out = predict_np_single(model, Xn)
    if kind == "age":
        pred = inv_age(out)
        r2, rmse, mae = metrics_reg(y_age_true.ravel(), pred)
        return {"R2": r2, "RMSE": rmse, "MAE": mae, "ACC": np.nan, "AUC": np.nan, "F1": np.nan}
    if kind == "mets":
        pred = inv_mets(out)
        r2, rmse, mae = metrics_reg(y_mets_true.ravel(), pred)
        return {"R2": r2, "RMSE": rmse, "MAE": mae, "ACC": np.nan, "AUC": np.nan, "F1": np.nan}
    if kind == "sex":
        acc, auc, f1v = metrics_sex_acc_auc_f1(y_sex_true, out, threshold=sex_threshold)
        return {"R2": np.nan, "RMSE": np.nan, "MAE": np.nan, "ACC": acc, "AUC": auc, "F1": f1v}
    raise ValueError(kind)


def train_single(model, kind, train_loader, val_loader, tag,
                 lr=1e-3, weight_decay=1e-6, epochs=220, metrics_every=10, sex_threshold=0.5):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    best_val = float("inf")
    best_state = None
    hist = []

    for ep in range(1, epochs + 1):
        tr_loss = run_epoch_single(model, train_loader, kind, optimizer=optimizer)
        va_loss = run_epoch_single(model, val_loader, kind, optimizer=None)
        scheduler.step(va_loss)

        if ep % metrics_every == 0 or ep == 1:
            m_tr = eval_single_metrics(model, kind, X_train_s, age_train, mets_train, sex_train, sex_threshold=sex_threshold)
            m_va = eval_single_metrics(model, kind, X_val_s, age_val, mets_val, sex_val, sex_threshold=sex_threshold)

            if kind in ("age", "mets"):
                print(f"[{tag}] ep={ep:03d} va_loss={va_loss:.4f} | TRAIN R2={m_tr['R2']:.3f} | VAL R2={m_va['R2']:.3f}")
            else:
                print(f"[{tag}] ep={ep:03d} va_loss={va_loss:.4f} | TRAIN AUC={m_tr['AUC']:.3f} | VAL AUC={m_va['AUC']:.3f}")

        hist.append((ep, tr_loss, va_loss, optimizer.param_groups[0]["lr"]))

        if va_loss < best_val - 1e-6:
            best_val = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    hist_df = pd.DataFrame(hist, columns=["epoch", "train_loss", "val_loss", "lr"])
    hist_df.to_csv(OUTDIR / f"train_history_single_{tag}.csv", index=False)
    print(f"[{tag}] best val loss: {best_val:.6f}")
    return hist_df

single_specs = [
    ("age", "single_TF_age_cont"),
    ("mets", "single_TF_MetSCORE"),
    ("sex", "single_TF_sex"),
]

single_test_rows = []

for kind, tag in single_specs:
    torch.manual_seed(SEED)    
    model_single = SingleOutTransformerNet(IN_DIM, 
                                           emb_dim=EMB_DIM, 
                                           nhead=NHEAD, 
                                           num_layers=NUM_LAYERS, 
                                           ff_dim=FF_DIM, 
                                           dropout=DROPOUT).to(DEVICE)    

    def _train_single_call():
        return train_single(
            model_single, kind,
            train_loader, val_loader,
            tag=tag,
            lr=LR, weight_decay=WD, epochs=EPOCHS,
            metrics_every=10,
            sex_threshold=0.5
        )

    hist_single, tr_time_s, pk_rss_mb, st_rss_mb = profile_training(_train_single_call, poll_every_sec=0.05)
    profile_rows.append({
        "model_tag": f"M3-single-output:{kind}",
        "train_seconds": tr_time_s,
        "start_rss_mb": st_rss_mb,
        "peak_rss_mb": pk_rss_mb,
        "rss_increase_mb": (pk_rss_mb - st_rss_mb) if (not np.isnan(pk_rss_mb) and not np.isnan(st_rss_mb)) else np.nan
    })
    print(f"[PROFILE {tag}] time={tr_time_s:.2f}s | start_rss={st_rss_mb:.1f}MB | peak_rss={pk_rss_mb:.1f}MB")

    torch.save(model_single.state_dict(), OUTDIR / f"model_{tag}.pt")

    m_te = eval_single_metrics(model_single, kind, X_test_s, age_test, mets_test, sex_test, sex_threshold=0.5)

    if kind in ("age", "mets"):
        single_test_rows.append(["single-output(M2)", "test", ("MetSCORE" if kind == "mets" else kind),
                                 m_te["R2"], m_te["RMSE"], m_te["MAE"], np.nan, np.nan, np.nan])
    else:
        single_test_rows.append(["single-output(M2)", "test", kind,
                                 np.nan, np.nan, np.nan, m_te["ACC"], m_te["AUC"], m_te["F1"]])

single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["model_type", "split", "output", "R2", "RMSE", "MAE", "ACC", "AUC", "F1"]
)
single_metrics_path = OUTDIR / "metrics_test_single_output_M3_age_MetSCORE_sex.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")
print("\nSaved:", single_metrics_path)
print(single_metrics_df)

profile_df = pd.DataFrame(profile_rows)
profile_path = OUTDIR / "training_time_memory_profile.csv"
profile_df.to_csv(profile_path, index=False, float_format="%.6f")
print("\nSaved:", profile_path)
print(profile_df)

# Optional comparison: M2 multi vs sum of M2 singles (time/memory)
def pick(tag):
    row = profile_df.loc[profile_df["model_tag"] == tag].copy()
    return None if row.empty else row.iloc[0]


m2_multi_prof = pick("M3_multi_transformer")
s_age = pick("M3-single-output:age")
s_mets = pick("M3-single-output:mets")
s_sex = pick("M3-single-output:sex")

if (m2_multi_prof is not None) and (s_age is not None) and (s_mets is not None) and (s_sex is not None):
    sum_single_time = float(s_age["train_seconds"] + s_mets["train_seconds"] + s_sex["train_seconds"])
    max_single_peak = float(np.nanmax([s_age["peak_rss_mb"], s_mets["peak_rss_mb"], s_sex["peak_rss_mb"]]))

    compare_df = pd.DataFrame([{
        "metric": "train_seconds",
        "M2_multi": float(m2_multi_prof["train_seconds"]),
        "sum_single_M2": sum_single_time
    }, {
        "metric": "peak_rss_mb",
        "M2_multi": float(m2_multi_prof["peak_rss_mb"]),
        "max_single_M2": max_single_peak
    }])

    compare_path = OUTDIR / "compare_M2_vs_sumSinglesM2_time_memory.csv"
    compare_df.to_csv(compare_path, index=False, float_format="%.6f")
    print("\nSaved:", compare_path)
    print(compare_df)

if not _HAS_PSUTIL:
    print("\n[Warning] psutil is not installed, so RSS memory profiling is NaN. Install with pip/conda to enable.")

/home/umberto/Escritorio/Simulations/Python/VirtualEnvironments/PythonVirtualEnvironment/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[single_TF_age_cont] ep=001 va_loss=0.2206 | TRAIN R2=0.520 | VAL R2=0.524
[single_TF_age_cont] ep=010 va_loss=0.1554 | TRAIN R2=0.696 | VAL R2=0.668
[single_TF_age_cont] ep=020 va_loss=0.1576 | TRAIN R2=0.747 | VAL R2=0.667
[single_TF_age_cont] best val loss: 0.151898
[PROFILE single_TF_age_cont] time=309.44s | start_rss=856.4MB | peak_rss=3372.5MB


/home/umberto/Escritorio/Simulations/Python/VirtualEnvironments/PythonVirtualEnvironment/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[single_TF_MetSCORE] ep=001 va_loss=0.1303 | TRAIN R2=0.736 | VAL R2=0.718
[single_TF_MetSCORE] ep=010 va_loss=0.0777 | TRAIN R2=0.854 | VAL R2=0.843
[single_TF_MetSCORE] ep=020 va_loss=0.0711 | TRAIN R2=0.883 | VAL R2=0.857
[single_TF_MetSCORE] best val loss: 0.070529
[PROFILE single_TF_MetSCORE] time=316.59s | start_rss=857.1MB | peak_rss=3317.6MB


/home/umberto/Escritorio/Simulations/Python/VirtualEnvironments/PythonVirtualEnvironment/lib/python3.12/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[single_TF_sex] ep=001 va_loss=0.1628 | TRAIN AUC=0.982 | VAL AUC=0.983
[single_TF_sex] ep=010 va_loss=0.1251 | TRAIN AUC=0.992 | VAL AUC=0.988
[single_TF_sex] ep=020 va_loss=0.1232 | TRAIN AUC=0.995 | VAL AUC=0.989
[single_TF_sex] best val loss: 0.123222
[PROFILE single_TF_sex] time=313.25s | start_rss=857.5MB | peak_rss=3344.7MB

Saved: Results_transformer/metrics_test_single_output_M3_age_MetSCORE_sex.csv
          model_type split    output        R2      RMSE       MAE       ACC  \
0  single-output(M2)  test       age  0.661886  7.841352  6.131313       NaN   
1  single-output(M2)  test  MetSCORE  0.822790  0.108446  0.070916       NaN   
2  single-output(M2)  test       sex       NaN       NaN       NaN  0.947302   

        AUC        F1  
0       NaN       NaN  
1       NaN       NaN  
2  0.988745  0.954887  

Saved: Results_transformer/training_time_memory_profile.csv
               model_tag  train_seconds  start_rss_mb  peak_rss_mb  \
0   M3_multi_transformer     314.309800 